# Adaptive Query Execution (AQE): Runtime Query Optimization

**Module**: 07_spark_sql_internals  
**Notebook**: 05_adaptive_query_execution.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Adaptive Query Execution re-optimizes query plans at runtime based on materialized stage statistics: coalescing shuffle partitions, switching join strategies, and optimizing skew joins.
- **Why does this topic exist?**: AQE eliminates static partition guessing and dynamically fixes data skew.
- **What problem does it solve?**: Static queries planned with outdated statistics before knowing actual filtered row counts.
- **Where is it used in real-world Data Engineering?**: Production performance tuning in Spark 3.x and 4.x.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Join strategies, shuffle, physical plans |
| **Required depth** | Working knowledge: can explain AQE's three main optimizations and when they apply |
| **Data Engineer** | Use AQE to reduce manual partition and join tuning. |
| **Senior Data Engineer** | Diagnose why AQE did or did not change a plan. |
| **Spark specialist** | Reason about runtime statistics, stage materialization, and AQE limits. |
| **Next topic** | `08_data_sources_and_formats/01_csv_and_json.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

AQE re-optimizes a query after shuffle stages materialize, when actual partition sizes are known. It coalesces
small partitions, can switch join strategies, and can split skewed partitions. AQE is enabled by default in modern
Spark, but it cannot fix every problem: it only sees shuffle boundaries, and some plans and join shapes are not
eligible.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Adaptive execution** | Runtime re-optimization | Uses actual shuffle statistics instead of static estimates |
| **Coalescing** | Merge small partitions | Reduces task count after a shuffle |
| **Join conversion** | Switch strategy at runtime | Sort-merge to broadcast when a side is small enough |
| **Skew join handling** | Split oversized partitions | Requires eligible join and skew thresholds |
| **Runtime statistics** | Actual partition sizes | Collected at shuffle boundaries |
| **Stage materialization** | AQE decision point | Optimization happens between stages |

### How it works, step by step

1. The query starts with an initial physical plan.
2. A shuffle stage runs and materializes partition statistics.
3. AQE reads the actual partition sizes.
4. It coalesces small partitions and may change the join strategy.
5. It can split skewed partitions for eligible joins.
6. The remaining stages execute with the revised plan.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. The 3 Pillars of AQE
1. **Dynamically Coalescing Shuffle Partitions**: Merges hundreds of tiny post-shuffle partitions into fewer optimal-sized partitions.
2. **Dynamically Switching Join Strategies**: Converts Sort-Merge Join to Broadcast Hash Join if the filtered build-side table turns out to be smaller than the broadcast threshold.
3. **Dynamically Handling Data Skew**: Automatically detects skewed partitions and splits them into smaller sub-partitions.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("AQE_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("=== Active AQE Configurations in Cluster ===")
print("spark.sql.adaptive.enabled:                    ", spark.conf.get("spark.sql.adaptive.enabled", "true"))
print("spark.sql.adaptive.coalescePartitions.enabled: ", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled", "true"))
print("spark.sql.adaptive.skewJoin.enabled:           ", spark.conf.get("spark.sql.adaptive.skewJoin.enabled", "true"))

---

## Experiment: Observing AdaptiveSparkPlan in Physical Plan

- **1. What are we testing?**: Verify that query plans are wrapped in AdaptiveSparkPlan.
- **2. What do we expect?**: AdaptiveSparkPlan operator appears at root of plan.

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
res = orders.groupBy("channel").count()
print("Physical plan with AQE:")
res.explain()

**3. What happened?**: AdaptiveSparkPlan operator is present.  
**4. Why did it happen?**: AQE is enabled in cluster defaults.  
**5. Practical Engineering Lesson**: AdaptiveSparkPlan re-evaluates physical operators at each shuffle stage boundary.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Variable daily data volume

- **Industry**: E-commerce
- **Scenario**: A daily job processes anywhere from 20 GB to 400 GB depending on promotions.
- **Problem**: A fixed shuffle partition count is too high on small days and too low on large days.
- **Why this concept is relevant**: AQE coalesces small partitions automatically, adapting the task count to the actual data size.
- **Implementation idea**: Keep AQE enabled, set a reasonable initial partition count, and compare task counts across days.
- **Expected result**: Stable runtime and task counts across variable volumes.
- **Alternative**: Schedule the partition count dynamically from upstream metadata.
- **Trade-offs**: Dynamic scheduling adds orchestration complexity; AQE handles the common case automatically.

### Use Case 2: Join that becomes broadcast after filtering

- **Industry**: Financial services
- **Scenario**: A dimension table is large overall but a selective filter leaves a small build side.
- **Problem**: Static statistics see the full table size and choose sort-merge, missing the opportunity to broadcast.
- **Why this concept is relevant**: AQE can convert the join after the filtered side is materialized and its actual size is known.
- **Implementation idea**: Inspect the AQE plan and confirm the join strategy changes at runtime.
- **Expected result**: A broadcast join with no large-side shuffle.
- **Alternative**: Add a broadcast hint after verifying the filtered size.
- **Trade-offs**: A hint is predictable but can become unsafe if the filter selectivity changes.

### Use Case 3: Skewed partition after a shuffle

- **Industry**: Telecom
- **Scenario**: One subscriber ID dominates a join, creating a single oversized reduce partition.
- **Problem**: Static planning cannot see the skew; AQE can detect oversized partitions at runtime.
- **Why this concept is relevant**: AQE skew join handling splits the oversized partition when the join shape is eligible.
- **Implementation idea**: Enable skew handling and verify the stage's max task duration falls.
- **Expected result**: A more balanced stage without manual salting.
- **Alternative**: Manual salting or hot-key isolation.
- **Trade-offs**: Manual techniques work for known keys but add code complexity; AQE is automatic but limited.

---

## Business Value

- Less manual partition tuning.
- Better performance across variable data volumes.
- Fewer skew-related incidents.
- Lower operational burden for recurring pipelines.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Keep AQE enabled unless you have a specific reason to disable it.
- Set `spark.sql.shuffle.partitions` to a sensible initial value; AQE coalesces from there.
- Check the AQE plan, not only the initial plan.
- Skew handling requires eligible joins and appropriate thresholds.
- AQE cannot optimize across arbitrary stage boundaries or non-shuffle operators.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: See AQE change the plan

**Steps**

1. Run a join with a selective filter on one side and a shuffle partition count much larger than needed.
2. Inspect the initial plan and the AQE plan in the UI or `explain()` output.
3. Compare task counts and join strategy before and after AQE.

**What to observe**: AQE coalesces small partitions and may convert the join to broadcast.

**Healthy result**: The runtime plan is more efficient than the static plan.

**Problematic result**: AQE does not apply because the join shape or plan stage is ineligible.

**Engineering takeaway**: AQE is powerful but bounded; verify it actually changed the plan.

---

## Trade-offs and Alternatives

### Advantages

- Automatic adaptation
- Reduces manual tuning
- Handles skew and small partitions

### Limitations

- Only at shuffle boundaries
- Not all joins eligible
- Can make plans harder to reproduce

### When to use it

- Variable workloads
- Join-heavy pipelines
- Skewed shuffles

### When not to use it

- Disabling AQE without cause
- Relying on it to fix bad layout
- Ignoring runtime plan changes

### Alternatives inside Spark

- Coalescing
- Join conversion
- Skew join handling

### Alternatives outside Spark

- Workload managers
- Adaptive warehouses
- Manual salting

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check AQE Enabled
**Problem Statement**: Verify spark.sql.adaptive.enabled.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Inspect Post-Shuffle Partition Coalescing
**Problem Statement**: Observe how AQE coalesced shuffle partitions in Spark UI SQL tab.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): AQE Skew Join Parameters
**Problem Statement**: Check skewedPartitionFactor setting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### AQE enabled but skew persists

**Scenario**: AQE is on, but one reduce task still takes far longer than the others.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the join type is eligible for skew handling.
2. Inspect the skew thresholds and partition sizes.
3. Consider manual salting or hot-key isolation if AQE cannot split the partition.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check AQE Enabled):
```python
print(spark.conf.get('spark.sql.adaptive.enabled'))
```

#### Solution 2 (Inspect Post-Shuffle Partition Coalescing):
```python
print('Inspected AQE nodes in localhost:4040/SQL')
```

#### Solution 3 (AQE Skew Join Parameters):
```python
print(spark.conf.get('spark.sql.adaptive.skewJoin.skewedPartitionFactor','5'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **AQE Revolution**: Re-optimizes plans at runtime using actual stage metrics.
- **Features**: Coalesce partitions, switch to broadcast join, split skewed partitions.

### Essential Technical Terms
- `AdaptiveSparkPlan`: Runtime adaptive query coordinator.
- `SkewJoin`: Splitting skewed partition chunks.

### Common Engineering Mistakes
- **Disabling AQE**: Forces reliance on static upfront estimates.

### Interview Questions on This Topic
- **Q: What are the three main features of Adaptive Query Execution?**
  - *A*: 1. Dynamically coalescing shuffle partitions, 2. Dynamically converting SortMergeJoin to BroadcastHashJoin, and 3. Dynamically splitting skewed join partitions.


### More Interview Questions

- **Q: What are the three main AQE features?**
  - *A*: Dynamically coalescing shuffle partitions, dynamically switching join strategies, and dynamically handling skew joins.
- **Q: Why does AQE need a shuffle boundary?**
  - *A*: It re-optimizes using actual partition statistics that only become available after a shuffle stage materializes.

### What to Study Next
Module 07 complete! Proceed to Module 08: Data Sources and File Formats.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')